# 带约束的露天矿坑极限问题 (CPIT)

**类别:** 装箱

来源: [https://www.hexaly.com/templates/constrained-pit-limit-cpit](https://www.hexaly.com/templates/constrained-pit-limit-cpit)


## 问题描述

在带约束的露天矿坑极限问题 (CPIT) 中,我们考虑一组可在若干时间段内从矿坑中开采的块。开采一个块需要消耗一定数量的资源。对于每种资源,在某个时间段内的资源消耗总和不能超过某个上限。每个块必须在其所有前驱块被开采的同一时间段或更晚的时间段开采。一个块的开采利润取决于该块和开采时间段。目标函数是最大化从块开采中获得的利润。

	

### 学到的要点

- 添加 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每个时间段的开采情况
- 通过 `[find](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#find)` 算子获取每个块开采时间段的索引
- 定义一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来对每个时间段的利润求和


## 数据

所提供的实例来自 [Espinoza 等人](https://link.springer.com/article/10.1007/s10479-012-1258-3),其格式如下:

- 一个 .cpit 文件存储与带约束的露天矿坑极限问题 (CPIT) 相关的数据:

- 块的数量
- 时间段的数量
- 资源的数量
- 折现率(用于计算块开采的利润)
- 对每种资源和每个时间段,该时间段内该资源消耗的上下界
- 对每个块,其利润
- 对每个块和每种资源,该块的开采对该资源的消耗(仅当非零时写入文件)
- 一个 .prec 文件存储每个块的前驱块。


## 程序

带约束的露天矿坑极限问题 (CPIT) 的 Hexaly 模型使用 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。每个集合表示在某个时间段内开采的块。我们添加一个虚拟时间段来表示保持未开采的块。借助 `[partition](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition)` 算子,我们确保每个块最多被开采一次。

对于给定类型的资源,一个 lambda 函数对所有被开采的块应用 `[sum](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#sum)` 算子来计算该时间段内该资源的总消耗量。注意,该求和中项的数量在搜索过程中是变化的,集合的大小也随之变化。然后我们可以根据为问题定义的上下界对该数量施加约束。

然后,使用 `[find](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#find)` 算子,我们可以获取每个块被选定的开采时间段的索引。这使我们能够写出块之间的优先关系约束:每个块必须在与其前驱块的同一时间段或更晚的时间段被开采。

一个时间段的利润通过一个 lambda 函数对该时间段内被开采的块应用 `[sum](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#sum)` 算子来计算。同样地,该求和中项的数量在搜索过程中是变化的,集合的大小也随之变化。我们最大化所有时间段的总利润。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def get_prec_file_name(instance_file):
    begin_index = instance_file.index("instances") + 10     # the size of "instances/" is 10
    end_index = instance_file.index(".cpit")
    return instance_file[0:begin_index] + "prec/" + \
            instance_file[begin_index:end_index] + ".prec"

def read_instance(instance_file):
    # Reading the first file containing constants of the problem
    with open(instance_file) as f:
        line = f.readline()     # Name of the instance
        line = f.readline()     # Type of problem
        line = f.readline().split(" ")  # Number of blocks
        nb_blocks = int(line[1])

        line = f.readline().split(" ")  # Number of periods
        nb_periods = int(line[1])

        line = f.readline().split(" ")  # Number of resources
        nb_resources = int(line[1])

        line = f.readline().split(" ")  # Discount rate
        discount_rate = float(line[1])

        # Resource Constraints Limits
        line = f.readline()  
        resource_LB,resource_UB = False, False
        resource_lower_bound = []
        resource_upper_bound = []
        for r in range(nb_resources):
            resource_lower_bound.append([])
            resource_upper_bound.append([])
            for t in range(nb_periods):
                line = f.readline().split(" ")
                if len(line) == 5:
                    resource_LB,resource_UB = True, True
                    resource_lower_bound[r].append(int(line[3]))
                    resource_upper_bound[r].append(int(line[4]))
                else:
                    if line[2] == "G":
                        resource_lower_bound[r].append(int(line[3]))
                        resource_LB = True
                    else: # line[2] == "L"
                        resource_upper_bound[r].append(int(line[3]))
                        resource_UB = True

        # Objective function
        line = f.readline()
        profit = [0 for b in range(nb_blocks)]
        for b in range(nb_blocks):
            line = f.readline().split(" ")
            profit[int(line[0])] = float(line[1])

        discounted_profit = [[profit[b]/pow(1+discount_rate,t) for b in range(nb_blocks)] \
                for t in range(nb_periods)]
        
        # Resource constraints coefficients
        # Some values are not defined and therefore equal 0
        # Here we index by resource r first, then the block index b
        line = f.readline()
        resource_use = [[0 for b in range(nb_blocks)] for r in range(nb_resources)]

        line = f.readline().split(" ")
        while line[0] != "EOF\n":
            resource_use[int(line[1])][int(line[0])] = float(line[2])
            line = f.readline().split(" ")
    
    # Reading the second file containing precedence relations
    with open(get_prec_file_name(instance_file)) as f:
        precedence = {}
        for b in range(nb_blocks):
            line = f.readline()[:-1].split(" ")
            if int(line[1]) > 0:
                precedence[int(line[0])] = [int(line[i]) for i in range(2,len(line))]
    
    return (nb_blocks, nb_periods, nb_resources, discount_rate, resource_LB, resource_UB, \
            resource_lower_bound, resource_upper_bound, discounted_profit, resource_use, precedence)

def main(instance_name, output_file, time_limit):
    nb_blocks, nb_periods, nb_resources, discount_rate, resource_LB, resource_UB, \
            resource_lower_bound, resource_upper_bound, discounted_profit, resource_use_data, \
            precedence = read_instance(instance_name)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model     # Declare the optimization model

        # Decision variables: period_vars[t] is the set of blocks extracted during period t
        # There is a dummy period for unextracted blocks
        period_vars = [model.set(nb_blocks) for t in range(nb_periods+1)]
        periods = model.array(period_vars)
        model.constraint(model.partition(periods))

        # Resources used to extract at period t must be within defined bounds
        resource_use = []
        for r in range(nb_resources):
            resource_use.append(model.array(resource_use_data[r]))
        consumption_lambda = [model.lambda_function(lambda b: resource_use[r][b]) \
                for r in range(nb_resources)]
        consumption_per_period = [[model.sum(period_vars[t],consumption_lambda[r]) \
                for t in range(nb_periods)]  for r in range(nb_resources)]

        for r in range(nb_resources):
            for t in range(nb_periods):
                if resource_LB:
                    model.constraint(consumption_per_period[r][t] >= resource_lower_bound[r][t])
                if resource_UB:
                    model.constraint(consumption_per_period[r][t] <= resource_upper_bound[r][t])

        # A block can only be extracted if all its predecessors have been extracted
        block_extraction_period = [model.find(periods, b) for b in range(nb_blocks)]
        for b in range(nb_blocks):
            if b in precedence.keys():
                for pred in precedence[b]:
                    model.constraint(block_extraction_period[b] >= block_extraction_period[pred])

        # Objective function: profit over extracted blocks
        profits = [model.array(discounted_profit[t]) for t in range(nb_periods)]
        profit_lambdas = [model.lambda_function(lambda b: profits[t][b]) for t in range(nb_periods)]
        profit_per_period = [model.sum(period_vars[t],profit_lambdas[t]) for t in range(nb_periods)]
        objective = model.sum(profit_per_period)

        # Maximizing the objective
        model.maximize(objective)

        model.close()

        # Parameterize
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        # Write the solution in a file following the format:
        # - 1st line: value of the objective (as an integer)
        # - following lines: (if the block is extracted) block's number, period's number
        # - "EOF" to signal the end of the file
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d\n" % int(objective.value))
                for b in range(nb_blocks):
                    if block_extraction_period[b].value < nb_periods:
                        f.write("{} {}\n".format(b, block_extraction_period[b].value))
                f.write("EOF")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python constrained_pit_limit.py instance_file \
            [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
